# 02 — Cross-Market Validation: Developed-Market Country ETF Surface

**Project:** Volatility & Dispersion (V&D) Cloud — personal research project
**Phase:** 2 — does the Phase 1 grammar travel to a new universe? (tickets G1–G9)
**Data:** public daily prices via `yfinance` only

**Why re-test on a new universe.** Phase 1 (`01_qqq_prototype_clean.ipynb`) built a coherent
regime grammar on a single, concentrated, mega-cap-heavy index. That is a weak basis for any
general claim. Phase 2 asks the more useful question: does the same construction — components,
signal variants, same-date cloud geometry, regime labels — produce a similarly coherent surface on
a structurally different universe, and if so, does *that* surface's forward-return structure
survive a skeptical challenge?

**Universe.** EFA (developed-market ex-US/Canada primary benchmark), URTH (broader global
developed-market context only — never a construction input), and 13 developed-market country ETFs,
equal-weighted, used to compute cross-sectional dispersion.

**Data.** Public daily adjusted prices via `yfinance`.

**Verified execution.** The revised code ran on real Yahoo data on October 13, 2026 (America/Los_Angeles), under Python 3.12.1 on Windows. All 22 tests and cached-data invariant checks passed; online/offline output hashes matched. This narrative was updated afterward without changing code cells. Result tables are exported in `results/`; interactive notebook outputs remain cleared.

## 1. Data Ingestion (G1)

Pull EFA, URTH, and the 13-country developed-market ETF universe. URTH is downloaded and retained
for later visual/context comparison only — it is explicitly excluded from the component return
matrix, cross-sectional dispersion, and every signal built below.

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt



REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").is_file())
sys.path.insert(0, str(REPO_ROOT))
import config
from src.vdcloud import (adjusted_close, price_returns, rolling_zscore, rolling_midpoint,
                         build_cloud, forward_return, purge_cross_boundary,
                         persistence_confirmed, prior_quantile_buckets, bucket_inference,
                         label_null_test, classify_zone)
from src.repro import download


PRIMARY_BENCHMARK = config.DM_PRIMARY_BENCHMARK   # EFA
CONTEXT_BENCHMARK = config.DM_CONTEXT_BENCHMARK   # URTH, context only
COMPONENT_TICKERS = config.DM_COUNTRY_ETF_UNIVERSE
ALL_TICKERS = [PRIMARY_BENCHMARK, CONTEXT_BENCHMARK] + COMPONENT_TICKERS

Z_WINDOW, VOL_WINDOW, SMOOTH_WINDOW = config.Z_WINDOW, config.VOL_WINDOW, config.SMOOTH_WINDOW
CLOUD_FAST, CLOUD_MEDIUM, CLOUD_SLOW = config.CONVERSION_WINDOW, config.BASE_WINDOW, config.SPAN_B_WINDOW
FORWARD_HORIZONS = config.FORWARD_HORIZONS
MIN_BUCKET_N = config.MIN_BUCKET_N

print(f"Primary benchmark : {PRIMARY_BENCHMARK}")
print(f"Context benchmark : {CONTEXT_BENCHMARK}  (context only -- excluded from all component math)")
print(f"Component universe: {len(COMPONENT_TICKERS)} developed-market country ETFs")

In [ ]:
raw = download(tickers=ALL_TICKERS, start=config.START_DATE, end=config.END_DATE,
                   auto_adjust=True, progress=True)

prices = adjusted_close(raw, ALL_TICKERS)
close_field = "Close (auto_adjust=True)"

returns = price_returns(prices)

efa_returns  = returns[PRIMARY_BENCHMARK].copy()
urth_returns = returns[CONTEXT_BENCHMARK].copy()   # context only -- never used in signal construction
dm_country_returns = returns[COMPONENT_TICKERS].copy()

assert PRIMARY_BENCHMARK not in dm_country_returns.columns
assert CONTEXT_BENCHMARK not in dm_country_returns.columns

# Equal-weight component return: row-wise mean, skipping NaNs -- the denominator
# adapts to however many country ETFs have valid data on a given date. Missing
# returns are excluded, never treated as zero.
dm_country_equal_weight_returns = dm_country_returns.mean(axis=1, skipna=True)

print(f"prices shape: {prices.shape}")
print(f"dm_country_returns shape: {dm_country_returns.shape}")
print(f"Date range: {prices.index[0].date()} to {prices.index[-1].date()}")

## 2. V&D Components (G2)

Same construction as Phase 1, applied to this universe:

- **Country-ETF cross-sectional dispersion** — std. dev. across the 13-ETF universe's daily
  returns, requiring at least 10 of 13 ETFs to have valid data on a given date.
- **EFA realized volatility** — rolling 21-day std. dev. of EFA daily returns, annualized.
- **URTH realized volatility** — computed in parallel, purely as a context series. It never feeds
  into any component, signal, or cloud calculation below.

Both core series are then rolling Z-scored (window = 126) and smoothed (5-day mean).

In [ ]:
MIN_COMPONENT_COUNT = 10
ANNUALIZATION_FACTOR = np.sqrt(252)

available_count = dm_country_returns.notna().sum(axis=1)
dm_country_dispersion = dm_country_returns.std(axis=1, skipna=True, ddof=1).where(available_count >= MIN_COMPONENT_COUNT)
dm_country_dispersion_ann = dm_country_dispersion * ANNUALIZATION_FACTOR

efa_realized_vol = efa_returns.rolling(window=VOL_WINDOW, min_periods=VOL_WINDOW).std(ddof=1) * ANNUALIZATION_FACTOR
# CONTEXT ONLY -- retained for visual comparison; not used in any signal below.
urth_realized_vol_context = urth_returns.rolling(window=VOL_WINDOW, min_periods=VOL_WINDOW).std(ddof=1) * ANNUALIZATION_FACTOR


# rolling_zscore imported from src.vdcloud


dm_country_dispersion_z = rolling_zscore(dm_country_dispersion_ann, Z_WINDOW)
efa_realized_vol_z = rolling_zscore(efa_realized_vol, Z_WINDOW)
urth_realized_vol_context_z = rolling_zscore(urth_realized_vol_context, Z_WINDOW)   # context only

dm_country_dispersion_z_smooth = dm_country_dispersion_z.rolling(window=SMOOTH_WINDOW, min_periods=SMOOTH_WINDOW).mean()
efa_realized_vol_z_smooth = efa_realized_vol_z.rolling(window=SMOOTH_WINDOW, min_periods=SMOOTH_WINDOW).mean()

corr = pd.DataFrame({"d": dm_country_dispersion_z_smooth, "v": efa_realized_vol_z_smooth}).corr().iloc[0, 1]
print(f"Correlation (dispersion_z_smooth vs efa_vol_z_smooth): {corr:.4f}")

## 3. The Four Signal Lenses (G3)

The same four-lens grammar as Phase 1, rebuilt from the two core smoothed components:

| Variant | Formula | Interpretation |
|---|---|---|
| Additive | `0.70 * dispersion_z + 0.30 * EFA_vol_z` | market temperature / turbulence lens |
| Penalty | `0.70 * dispersion_z - 0.30 * EFA_vol_z` | constructive-dispersion / regime-quality lens |
| Dispersion-only | `dispersion_z` | pure cross-market differentiation lens |
| Volatility-only | `EFA_vol_z` | pure EFA stress/noise lens |

URTH's context Z-score is not referenced in any of these four formulas.

In [ ]:
DISPERSION_WEIGHT, VOLATILITY_WEIGHT = 0.70, 0.30

vd_additive_signal = DISPERSION_WEIGHT * dm_country_dispersion_z_smooth + VOLATILITY_WEIGHT * efa_realized_vol_z_smooth
vd_penalty_signal  = DISPERSION_WEIGHT * dm_country_dispersion_z_smooth - VOLATILITY_WEIGHT * efa_realized_vol_z_smooth
vd_dispersion_only_signal = dm_country_dispersion_z_smooth.copy()
vd_volatility_only_signal = efa_realized_vol_z_smooth.copy()

signal_variants = {
    "additive": vd_additive_signal,
    "penalty": vd_penalty_signal,
    "dispersion_only": vd_dispersion_only_signal,
    "volatility_only": vd_volatility_only_signal,
}
for name, s in signal_variants.items():
    print(f"  {name:18s} valid obs: {s.notna().sum()}")

corr_matrix = pd.DataFrame(signal_variants).corr()
corr_matrix

## 4. Cloud Geometry and Regime Labels (G4)

Same-date Ichimoku-style cloud geometry (9/26/52-day rolling midpoints), applied independently to
each of the four signal variants, producing `above_cloud` / `inside_cloud` / `below_cloud` labels
for each lens. No forward shifting anywhere in this construction.

In [ ]:
# rolling_midpoint imported from src.vdcloud


def build_same_date_cloud_regime(signal, fast=CLOUD_FAST, medium=CLOUD_MEDIUM, slow=CLOUD_SLOW):
    return build_cloud(signal, fast, medium, slow).dropna(subset=["regime_state"])


cloud_regimes = {name: build_same_date_cloud_regime(s) for name, s in signal_variants.items()}

for name, frame in cloud_regimes.items():
    counts = frame["regime_state"].value_counts()
    print(f"{name:18s} above={counts.get('above_cloud', 0):5d}  inside={counts.get('inside_cloud', 0):5d}  below={counts.get('below_cloud', 0):5d}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
additive_frame = cloud_regimes["additive"]
ax.fill_between(additive_frame.index, additive_frame["cloud_bottom"], additive_frame["cloud_top"],
                 color="lightsteelblue", alpha=0.45, label="Cloud (Span A / Span B)")
ax.plot(additive_frame.index, additive_frame["signal"], color="tab:blue", linewidth=1.1, label="Additive signal")
ax.axhline(0, color="gray", linewidth=0.6, linestyle=":")
ax.set_title("V&D Cloud | DM Country ETF Surface — Additive Signal with Same-Date Cloud")
ax.legend(loc="upper left"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

latest_date = additive_frame.index.max()
print("Latest same-date regime read (diagnostic snapshot, not a forecast):")
for name, frame in cloud_regimes.items():
    if latest_date in frame.index:
        print(f"  {name:18s}: {frame.loc[latest_date, 'regime_state']}")

## 5. Forward-Return Findings (G6)

For each lens, regime, and forward horizon, compute descriptive EFA forward-return statistics and
the excess of the regime-conditional mean versus the unconditional mean over the same horizon.
This is a **descriptive** pass only — no baseline comparison, no significance testing yet. Those
come next.

In [ ]:
def compute_forward_return(return_series, horizon):
    return forward_return(return_series, horizon)


efa_fwd = pd.DataFrame({f"fwd_{h}d": compute_forward_return(efa_returns, h) for h in FORWARD_HORIZONS})
print(f"efa_fwd shape: {efa_fwd.shape}")

In [ ]:
regime_forward_rows = []
for lens_name, frame in cloud_regimes.items():
    joined = frame[["regime_state"]].join(efa_fwd, how="left")
    for h in FORWARD_HORIZONS:
        col = f"fwd_{h}d"
        uncond_mean = joined[col].dropna().mean()
        for regime, group in joined.dropna(subset=["regime_state", col]).groupby("regime_state"):
            r = group[col]
            regime_forward_rows.append({
                "variant": lens_name, "regime": regime, "horizon": h, "count": len(r),
                "mean_return": r.mean(), "excess_vs_unconditional": r.mean() - uncond_mean,
            })

regime_forward_return_summary = pd.DataFrame(regime_forward_rows)

top_buckets = regime_forward_return_summary.sort_values("excess_vs_unconditional", ascending=False).head(10)
bottom_buckets = regime_forward_return_summary.sort_values("excess_vs_unconditional", ascending=True).head(10)
print("--- Top 10 descriptive EFA excess-return buckets ---")
print(top_buckets.to_string(index=False))
print("\n--- Bottom 10 ---")
print(bottom_buckets.to_string(index=False))

This grid has 4 variants × 3 regimes × 4 horizons = 48 descriptive cells. Searching for extreme excess returns creates selection risk. The current table is exported with the verified execution record.


## 6. Baseline Challenge (G7)

A more skeptical follow-up: do the V&D regime buckets look any different from much simpler,
well-known baseline classifiers — realized-volatility quartile, country-dispersion quartile, EFA
trend state (price vs. 200-day moving average), and drawdown buckets (cut at
-5% / -15% from a running maximum)? If a baseline classifier produces forward-return spreads just
as large as the V&D spreads, the V&D buckets may simply be repackaging information a much simpler
classifier already captures.

In [ ]:
efa_price = prices[PRIMARY_BENCHMARK]

realized_vol_quartile = prior_quantile_buckets(efa_realized_vol, ["vol_q1_low", "vol_q2", "vol_q3", "vol_q4_high"])
dispersion_quartile = prior_quantile_buckets(dm_country_dispersion_ann, ["disp_q1_low", "disp_q2", "disp_q3", "disp_q4_high"])

sma_200 = efa_price.rolling(window=200, min_periods=200).mean()
efa_trend_state = pd.Series(np.where(efa_price > sma_200, "uptrend", "downtrend"), index=efa_price.index).where(sma_200.notna())

running_max = efa_price.cummax()
drawdown = efa_price / running_max - 1.0
drawdown_bucket = pd.cut(drawdown, bins=[-1.0, -0.15, -0.05, 0.0], labels=["dd_deep", "dd_moderate", "dd_shallow"])

baseline_classifiers = {
    "realized_vol_quartile": realized_vol_quartile,
    "dispersion_quartile": dispersion_quartile,
    "efa_trend_state": efa_trend_state,
    "drawdown_bucket": drawdown_bucket,
}
print(f"Baseline classifiers built: {list(baseline_classifiers.keys())}")

In [ ]:
# Evaluate all classifiers on the same available dates.
common_index = efa_fwd.index
valid_common = efa_fwd["fwd_21d"].notna()
for labels in baseline_classifiers.values():
    valid_common &= labels.reindex(common_index).notna()
for cloud in cloud_regimes.values():
    valid_common &= cloud["regime_state"].reindex(common_index).notna()
common_index = common_index[valid_common]
baseline_rows = []
for clf_name, clf_series in baseline_classifiers.items():
    joined = pd.DataFrame({"bucket": clf_series.reindex(common_index)}).join(efa_fwd[["fwd_21d"]], how="left")
    uncond_mean = joined["fwd_21d"].dropna().mean()
    for bucket, group in joined.dropna(subset=["bucket", "fwd_21d"]).groupby("bucket", observed=True):
        r = group["fwd_21d"]
        baseline_rows.append({
            "classifier": clf_name, "bucket": bucket, "count": len(r),
            "mean_return": r.mean(), "excess_vs_unconditional": r.mean() - uncond_mean,
        })

baseline_forward_return_summary = pd.DataFrame(baseline_rows)
baseline_spread_by_classifier = (
    baseline_forward_return_summary.groupby("classifier")["excess_vs_unconditional"]
    .agg(lambda s: s.max() - s.min())
)
vd_spread_21d = pd.Series({name: cloud["regime_state"].reindex(common_index).to_frame()
    .join(efa_fwd[["fwd_21d"]]).groupby("regime_state")["fwd_21d"].mean().pipe(lambda s: s.max()-s.min())
    for name, cloud in cloud_regimes.items()})

print("21D excess spread, baseline classifiers:")
print(baseline_spread_by_classifier)
print("\n21D excess spread, V&D lenses:")
print(vd_spread_21d)

**Verified descriptive comparison:** drawdown and realized-volatility-quartile spreads are 2.637 and 2.016 percentage points; the largest V&D spread is 1.077 points (volatility-only). All use common 21-day dates. No incremental baseline-conditioned effect is tested.


## 7. Exploratory HAC and Label-Null Checks

The 13 highlighted variant/regime/horizon buckets were selected in the earlier private research after inspecting descriptive findings. They are fixed here for continuity, not independent prespecified hypotheses.

Shuffled labels provide an exploratory null. In the reviewer's no-information simulation with persistent labels and overlapping outcomes, it produced excessive false positives (see `scripts/null_calibration.py`). That illustrative simulation is not a calibration of the selected real-data bucket count. Circular shifts (at least one horizon of offset) add a dependence-preserving sensitivity check, subject to approximate stationarity and a wrap boundary. Neither corrects selection or multiple testing. Both use 1,000 seeded draws and plus-one p-values.

Chronological OLS uses an intercept and bucket indicator, with HAC `maxlags = min(horizon - 1, 63)`. A reported crossing of shuffled-label p ≤ 0.10 and |HAC t| ≥ 1.65, in a bucket with at least 30 observations, is a heuristic diagnostic, not statistical validation. Report the circular-shift result alongside it.


In [ ]:
def run_regime_label_permutation_test(frame, regime_col, return_col, target_regime, n_perm=1000, random_seed=42):
    return label_null_test(frame, regime_col, return_col, target_regime, n_perm, random_seed)

print("Permutation helper defined.")

In [ ]:
try:
    import statsmodels.api as sm
    STATSMODELS_AVAILABLE = True
except ImportError:
    STATSMODELS_AVAILABLE = False


def run_hac_bucket_regression(frame, regime_col, target_regime, return_col, maxlags):
    return bucket_inference(frame, regime_col, return_col, target_regime, maxlags)

print(f"statsmodels available: {STATSMODELS_AVAILABLE}")

In [ ]:
PERMUTATION_P_THRESHOLD = config.PERMUTATION_TWO_SIDED_P   # 0.10
HAC_T_STAT_THRESHOLD = config.HAC_TSTAT_THRESHOLD           # 1.65

# The original private research (ticket G8) hand-picked these exact 13 buckets --
# the top 8 strongest-positive-excess G6 buckets, 4 weakest/contrast buckets, and
# 1 bucket matching each lens's then-current live regime state. Reproduced here
# verbatim (not re-derived from this run's own top/bottom buckets) so this notebook
# tests the same specific claims the original research made.
key_bucket_spec = [
    ("volatility_only", "above_cloud", 21),
    ("volatility_only", "above_cloud", 63),
    ("penalty", "below_cloud", 21),
    ("penalty", "below_cloud", 63),
    ("additive", "above_cloud", 21),
    ("additive", "above_cloud", 63),
    ("dispersion_only", "below_cloud", 21),
    ("dispersion_only", "below_cloud", 63),
    ("volatility_only", "inside_cloud", 21),
    ("additive", "inside_cloud", 21),
    ("dispersion_only", "above_cloud", 21),
    ("penalty", "above_cloud", 21),
    ("volatility_only", "below_cloud", 21),
]
key_buckets = pd.DataFrame(key_bucket_spec, columns=["variant", "regime", "horizon"])

robustness_rows = []
for _, row in key_buckets.iterrows():
    frame = cloud_regimes[row["variant"]][["regime_state"]].reindex(efa_fwd.index).join(efa_fwd, how="left")
    return_col = f"fwd_{row['horizon']}d"
    maxlags = min(int(row["horizon"]) - 1, 63)

    perm = run_regime_label_permutation_test(frame, "regime_state", return_col, row["regime"],
                                              n_perm=1000, random_seed=42)
    circular = label_null_test(frame, "regime_state", return_col, row["regime"],
                              n_perm=1000, random_seed=42, method="circular_shift", min_shift=int(row["horizon"]))
    hac = run_hac_bucket_regression(frame, "regime_state", row["regime"], return_col, maxlags)

    perm_interesting = perm["bucket_n"] >= MIN_BUCKET_N and perm["empirical_two_sided_p_value"] <= PERMUTATION_P_THRESHOLD
    # The minimum bucket size applies to both thresholds, matching results/headline_summary.json.
    hac_interesting = (perm["bucket_n"] >= MIN_BUCKET_N and not np.isnan(hac["hac_t_stat"])
                       and abs(hac["hac_t_stat"]) >= HAC_T_STAT_THRESHOLD)
    verdict = "survives initial robustness" if (perm_interesting and hac_interesting) else \
              "permutation-interesting but HAC-weak" if perm_interesting else \
              "HAC-interesting but permutation-ordinary" if hac_interesting else "weak / likely noise"

    robustness_rows.append({
        "variant": row["variant"], "regime": row["regime"], "horizon": row["horizon"],
        "bucket_n": perm["bucket_n"], "empirical_p": perm["empirical_two_sided_p_value"],
        "hac_t_stat": hac["hac_t_stat"], "circular_shift_p": circular["empirical_two_sided_p_value"], "robustness_verdict": verdict,
    })

robustness_verdict = pd.DataFrame(robustness_rows)
print(robustness_verdict.to_string(index=False))
print(f"\nSurvived both bars: {(robustness_verdict['robustness_verdict'] == 'survives initial robustness').sum()} of {len(robustness_verdict)}")

**Verified revised result:** 9/13 shuffled-label crossings, 1/13 HAC crossings, 0/13 circular-shift crossings. Additive / inside_cloud / 21-day is the sole HAC crossing (t = −1.732178), with circular-shift p = 0.160839. Prior selection and multiplicity are uncorrected. The illustrative null calibration does not establish an expected discovery count for this real-data grid.


## 8. Phase 2 Verdict — Verified Revised Run

Among thirteen previously selected buckets, nine cross the exploratory shuffled-label p ≤ 0.10 threshold, one crosses |HAC t| ≥ 1.65, and none crosses the circular-shift p ≤ 0.10 sensitivity threshold. The isolated HAC crossing (additive / inside_cloud / 21-day, t = −1.732178) has circular-shift p = 0.160839. The legacy shuffle-plus-HAC verdict field does not imply it passed the circular-shift check.

On common 21-day dates, drawdown and realized-volatility baseline spreads exceed the largest V&D-lens spread. This compares spreads; it does not test incremental information conditional on baselines. No predictive validity or consistent baseline advantage is established. Selection, multiplicity, overlapping outcomes, and the circular-shift stationarity assumption remain limitations.
